# E-Commerce Order Analytics — Data Warehouse ETL Pipeline
## Final Project: Databricks Structured Streaming + Delta Tables

---

## Star Schema Architecture

**Fact Table:** `fact_orders` (center) connected to 4 dimension tables:
- `fact_orders.customer_id` → `dim_customer.customer_id` (PK)
- `fact_orders.product_id` → `dim_product.product_id` (PK)
- `fact_orders.date_key` → `dim_date.date_key` (PK)
- `fact_orders.location_key` → `dim_location.country_code` (PK)

## Key Relationships

| Table | Key Column | Key Type | Description |
|---|---|---|---|
| `fact_orders` | `order_id` | Primary Key | Unique order identifier |
| `fact_orders` | `customer_id` | Foreign Key → `dim_customer` | Links order to customer |
| `fact_orders` | `product_id` | Foreign Key → `dim_product` | Links order to product |
| `fact_orders` | `date_key` | Foreign Key → `dim_date` | Links order to calendar date |
| `fact_orders` | `location_key` | Foreign Key → `dim_location` | Links order to geography |
| `dim_customer` | `customer_id` | Primary Key | Unique customer identifier |
| `dim_product` | `product_id` | Primary Key | Unique product identifier |
| `dim_date` | `date_key` | Primary Key | Integer date key (YYYYMMDD) |
| `dim_location` | `country_code` | Primary Key | ISO 2-letter country code |

## Data Sources — Final Mapping

| Source | Technology | Delta Table |
|---|---|---|
| Relational DB (OLTP) | SQLite simulating Azure SQL | `dim_customer` |
| NoSQL Document Store | MongoDB Atlas | `dim_product` |
| Cloud File System | CSV from Databricks Volume | `dim_location` |
| Derived Pipeline | PySpark calendar generation | `dim_date` |
| Streaming File Source | JSON files via Spark AutoLoader | `fact_orders` (Bronze) |

---
## ⚙️ STEP 0 — Environment Setup

Install the MongoDB connector for Spark and define all configuration constants.

> **Note:** `%pip install` runs once per cluster session. After installing, the cluster does **not** need to be restarted — Databricks handles this automatically in a notebook context.


In [0]:
%pip install pymongo dnspython

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
MONGO_URI = "mongodb+srv://qad4ya_db_user:<My Password>@cluster0.lp3tmgb.mongodb.net/?appName=Cluster0"
MONGO_DB  = "ecommerce_dw"   


ORDERS_CSV_PATH  = "/Volumes/workspace/default/orders_data/"
DELTA_BASE_PATH  = "/Volumes/workspace/default/orders_data/delta/"      


DELTA_CUSTOMER = DELTA_BASE_PATH + "dim_customer"
DELTA_PRODUCT  = DELTA_BASE_PATH + "dim_product"
DELTA_DATE     = DELTA_BASE_PATH + "dim_date"
DELTA_LOCATION = DELTA_BASE_PATH + "dim_location"
DELTA_ORDERS   = DELTA_BASE_PATH + "fact_orders"

CHECKPOINT_PATH  = "/Volumes/workspace/default/orders_data/checkpoints/fact_orders"

print("✅ Configuration loaded.")
print(f"   MongoDB DB  : {MONGO_DB}")
print(f"   Orders CSV  : {ORDERS_CSV_PATH}")
print(f"   Delta root  : {DELTA_BASE_PATH}")

✅ Configuration loaded.
   MongoDB DB  : ecommerce_dw
   Orders CSV  : /Volumes/workspace/default/orders_data/
   Delta root  : /Volumes/workspace/default/orders_data/delta/


---
## 🌱 STEP 1 — Seed Dimension Data into MongoDB Atlas

Dimension tables (`dim_customer`, `dim_product`, `dim_location`) represent **slowly-changing reference data** — they describe the *who*, *what*, and *where* of business events.

We store them in **MongoDB Atlas** (NoSQL / document store) because:
- Dimension records have flexible, nested structures (e.g. product attributes may vary by category)
- Atlas provides a managed, cloud-hosted NoSQL service accessible from any Databricks cluster
- This mirrors real-world architectures where reference data lives in an operational database

This cell inserts the dimension records once. Re-running it is safe — it uses `replace_one` with `upsert=True` to avoid duplicates.


In [0]:
from pymongo import MongoClient

client = MongoClient(MONGO_URI)
db     = client[MONGO_DB]

# ── dim_customer ──────────────────────────────────────────────
customers = [
    {"customer_id": 1, "name": "Alice Johnson", "email": "alice@example.com", "country_code": "US"},
    {"customer_id": 2, "name": "Bob Smith",     "email": "bob@example.com",   "country_code": "GB"},
    {"customer_id": 3, "name": "Chen Wei",       "email": "chen@example.com",  "country_code": "CN"},
    {"customer_id": 4, "name": "Maria Garcia",  "email": "maria@example.com", "country_code": "MX"},
    {"customer_id": 5, "name": "Yuki Tanaka",   "email": "yuki@example.com",  "country_code": "JP"},
]
for c in customers:
    db.dim_customer.replace_one({"customer_id": c["customer_id"]}, c, upsert=True)
print(f"✅ dim_customer: {db.dim_customer.count_documents({})} documents in Atlas")

# ── dim_product ───────────────────────────────────────────────
products = [
    {"product_id": 1, "product_name": "Wireless Headphones", "category": "Electronics", "unit_price": 79.99},
    {"product_id": 2, "product_name": "Python Cookbook",     "category": "Books",       "unit_price": 39.99},
    {"product_id": 3, "product_name": "Yoga Mat",            "category": "Sports",      "unit_price": 25.99},
    {"product_id": 4, "product_name": "Coffee Grinder",      "category": "Kitchen",     "unit_price": 49.99},
    {"product_id": 5, "product_name": "USB-C Hub",           "category": "Electronics", "unit_price": 35.99},
]
for p in products:
    db.dim_product.replace_one({"product_id": p["product_id"]}, p, upsert=True)
print(f"✅ dim_product:  {db.dim_product.count_documents({})} documents in Atlas")

# ── dim_location ──────────────────────────────────────────────
locations = [
    {"country_code": "US", "country_name": "United States", "region": "Americas", "capital": "Washington D.C."},
    {"country_code": "GB", "country_name": "United Kingdom", "region": "Europe",  "capital": "London"},
    {"country_code": "CN", "country_name": "China",          "region": "Asia",    "capital": "Beijing"},
    {"country_code": "MX", "country_name": "Mexico",         "region": "Americas","capital": "Mexico City"},
    {"country_code": "JP", "country_name": "Japan",          "region": "Asia",    "capital": "Tokyo"},
]
for l in locations:
    db.dim_location.replace_one({"country_code": l["country_code"]}, l, upsert=True)
print(f"✅ dim_location: {db.dim_location.count_documents({})} documents in Atlas")

client.close()
print("\n✅ All dimension data seeded into MongoDB Atlas successfully.")

✅ dim_customer: 5 documents in Atlas
✅ dim_product:  5 documents in Atlas
✅ dim_location: 5 documents in Atlas

✅ All dimension data seeded into MongoDB Atlas successfully.


---
## 📤 STEP 2 — Extract Dimension Tables from MongoDB Atlas → Spark DataFrames

We read the three dimension collections from Atlas using `pymongo` and convert them to **PySpark DataFrames**.

**Why convert to Spark DataFrames?**
- Enables Spark SQL joins with the streaming fact table
- Allows Delta Table writes using the same unified API
- Scales to large dimension tables if needed

The `_id` column (MongoDB internal ObjectId) is dropped — it has no meaning in our analytical schema.


In [0]:
from pymongo import MongoClient
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("EcommerceETL").getOrCreate()

client = MongoClient(MONGO_URI)
db     = client[MONGO_DB]

def mongo_to_df(collection_name, drop_cols=["_id"]):
    """Fetch all documents from a MongoDB collection, return as Spark DataFrame."""
    docs = list(db[collection_name].find({}, {"_id": 0}))  # exclude _id at query time
    return spark.createDataFrame(docs)

df_customer = mongo_to_df("dim_customer")
df_product  = mongo_to_df("dim_product")
df_location = mongo_to_df("dim_location")

client.close()

print("✅ Dimension DataFrames extracted from MongoDB Atlas:")
print(f"   dim_customer : {df_customer.count()} rows")
print(f"   dim_product  : {df_product.count()} rows")
print(f"   dim_location : {df_location.count()} rows")

print("\n── dim_customer schema ──")
df_customer.printSchema()

print("── dim_product schema ──")
df_product.printSchema()

print("── dim_location schema ──")
df_location.printSchema()

✅ Dimension DataFrames extracted from MongoDB Atlas:
   dim_customer : 5 rows
   dim_product  : 5 rows
   dim_location : 5 rows

── dim_customer schema ──
root
 |-- country_code: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- email: string (nullable = true)
 |-- name: string (nullable = true)

── dim_product schema ──
root
 |-- category: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- unit_price: double (nullable = true)

── dim_location schema ──
root
 |-- capital: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- country_name: string (nullable = true)
 |-- region: string (nullable = true)



---
## 🔄 STEP 3 — Transform & Load Dimension Tables → Delta Tables

**What is a Delta Table?**
Delta Lake is an open-source storage layer that brings ACID transactions to Apache Spark. Delta Tables:
- Support `INSERT`, `UPDATE`, `DELETE`, and `MERGE` operations on big data
- Store data in Parquet format with a `_delta_log/` transaction log
- Enable **Time Travel** (query historical versions)
- Are the required target format for Structured Streaming

We also derive `dim_date` **in this pipeline** from the order dates — no external source needed.
The `date_key` is an integer in `YYYYMMDD` format (e.g. `20240115`) — a standard data warehouse pattern that makes date-range filtering fast.


In [0]:
from pyspark.sql.functions import (
    col, lit, year, month, dayofweek, quarter,
    date_format, to_date, expr, when
)
from pyspark.sql.types import IntegerType
from delta.tables import DeltaTable

# ── Write dim_customer ────────────────────────────────────────
(
    df_customer
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DELTA_CUSTOMER)
)
print(f"✅ dim_customer written to Delta: {DELTA_CUSTOMER}")
spark.read.format("delta").load(DELTA_CUSTOMER).show(truncate=False)

# ── Write dim_product ─────────────────────────────────────────
(
    df_product
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DELTA_PRODUCT)
)
print(f"✅ dim_product written to Delta: {DELTA_PRODUCT}")
spark.read.format("delta").load(DELTA_PRODUCT).show(truncate=False)

# ── Write dim_location ────────────────────────────────────────
(
    df_location
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DELTA_LOCATION)
)
print(f"✅ dim_location written to Delta: {DELTA_LOCATION}")
spark.read.format("delta").load(DELTA_LOCATION).show(truncate=False)

✅ dim_customer written to Delta: /Volumes/workspace/default/orders_data/delta/dim_customer
+------------+-----------+-----------------+-------------+
|country_code|customer_id|email            |name         |
+------------+-----------+-----------------+-------------+
|US          |1          |alice@example.com|Alice Johnson|
|GB          |2          |bob@example.com  |Bob Smith    |
|CN          |3          |chen@example.com |Chen Wei     |
|MX          |4          |maria@example.com|Maria Garcia |
|JP          |5          |yuki@example.com |Yuki Tanaka  |
+------------+-----------+-----------------+-------------+

✅ dim_product written to Delta: /Volumes/workspace/default/orders_data/delta/dim_product
+-----------+----------+-------------------+----------+
|category   |product_id|product_name       |unit_price|
+-----------+----------+-------------------+----------+
|Electronics|1         |Wireless Headphones|79.99     |
|Books      |2         |Python Cookbook    |39.99     |
|Sports 

In [0]:
# ── Build and write dim_date ──────────────────────────────────
# dim_date is derived entirely within the pipeline.
# We generate a date range covering all possible order dates.

from pyspark.sql.functions import sequence, explode, to_date, expr
import pyspark.sql.functions as F

# Generate every calendar date from 2024-01-01 to 2024-12-31
date_range = spark.sql("""
    SELECT explode(sequence(
        to_date('2024-01-01'),
        to_date('2024-12-31'),
        interval 1 day
    )) AS full_date
""")

df_date = (
    date_range
    .withColumn("date_key",    F.date_format("full_date", "yyyyMMdd").cast(IntegerType()))
    .withColumn("year",        F.year("full_date"))
    .withColumn("quarter",     F.quarter("full_date"))
    .withColumn("month",       F.month("full_date"))
    .withColumn("month_name",  F.date_format("full_date", "MMMM"))
    .withColumn("day_of_week", F.date_format("full_date", "EEEE"))
    # is_weekend: 1 if Saturday(7) or Sunday(1) in Spark's dayofweek (1=Sun, 7=Sat)
    .withColumn("is_weekend",
        F.when(F.dayofweek("full_date").isin(1, 7), 1).otherwise(0)
    )
    .withColumn("full_date", F.date_format("full_date", "yyyy-MM-dd"))  # store as string
    .select("date_key", "full_date", "year", "quarter", "month",
            "month_name", "day_of_week", "is_weekend")
)

(
    df_date
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DELTA_DATE)
)
print(f"✅ dim_date written to Delta: {DELTA_DATE}")
print(f"   Total date records: {df_date.count()} (full year 2024)")
df_date.filter("month = 1").show(5, truncate=False)

✅ dim_date written to Delta: /Volumes/workspace/default/orders_data/delta/dim_date
   Total date records: 366 (full year 2024)
+--------+----------+----+-------+-----+----------+-----------+----------+
|date_key|full_date |year|quarter|month|month_name|day_of_week|is_weekend|
+--------+----------+----+-------+-----+----------+-----------+----------+
|20240101|2024-01-01|2024|1      |1    |January   |Monday     |0         |
|20240102|2024-01-02|2024|1      |1    |January   |Tuesday    |0         |
|20240103|2024-01-03|2024|1      |1    |January   |Wednesday  |0         |
|20240104|2024-01-04|2024|1      |1    |January   |Thursday   |0         |
|20240105|2024-01-05|2024|1      |1    |January   |Friday     |0         |
+--------+----------+----+-------+-----+----------+-----------+----------+
only showing top 5 rows


---
## [Note] Preliminary CSV Streaming Demo — Not the Final Rubric Pipeline

> This section demonstrates an earlier CSV-based streaming approach for reference only.
> The final rubric-compliant pipeline (AutoLoader + JSON files + Bronze/Silver/Gold) begins in the section below.

---

## STEP 4 — Structured Streaming: Read Orders CSV as a Real-Time Stream

### What is Spark Structured Streaming?

Structured Streaming treats a data source as an **unbounded, continuously growing table**. New rows appear as new files land in the source folder (or as new Kafka messages, etc.). Spark processes each new batch of data as it arrives — this is called **micro-batch processing**.

```
  orders/ folder on DBFS
  ┌────────────────────────────────────────┐
  │  orders.csv  ←── existing file         │
  │  orders_new.csv ←── NEW file arrives   │  ──►  Spark detects new file
  │  ...                                   │       processes only new rows
  └────────────────────────────────────────┘       writes to Delta Table
```

Key differences from a batch read:
| Feature | Batch (`spark.read`) | Streaming (`spark.readStream`) |
|---|---|---|
| Reads new files automatically | ❌ | ✅ |
| Requires trigger/checkpoint | ❌ | ✅ |
| Output must be a sink (Delta, Kafka, etc.) | ❌ | ✅ |
| `.show()` works directly | ✅ | ❌ (must use `writeStream`) |

### Upload orders.csv before running this step
1. In Databricks: **Data** → **Add Data** → **Upload File**
2. Select your `orders.csv`
3. Set DBFS path to `/FileStore/tables/orders/orders.csv`
4. Click **Upload**


In [0]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DateType
)
import pyspark.sql.functions as F

# ── Define the schema of the incoming CSV ─────────────────────
# Structured Streaming requires an explicit schema — it cannot infer
# schema from a stream because it doesn't read the whole file upfront.
orders_schema = StructType([
    StructField("order_id",    IntegerType(), nullable=False),
    StructField("customer_id", IntegerType(), nullable=False),
    StructField("product_id",  IntegerType(), nullable=False),
    StructField("quantity",    IntegerType(), nullable=False),
    StructField("order_date",  StringType(),  nullable=False),
    StructField("status",      StringType(),  nullable=False),
])

# ── Create the streaming DataFrame ───────────────────────────
# readStream.format("csv") monitors the folder for new CSV files.
# latestFirst=False ensures files are processed in arrival order.
stream_raw = (
    spark
    .readStream
    .format("csv")
    .option("header", "true")
    .option("latestFirst", "false")
    .schema(orders_schema)
    .load(ORDERS_CSV_PATH)
)

print("✅ Streaming DataFrame created. Schema:")
stream_raw.printSchema()
print(f"   isStreaming = {stream_raw.isStreaming}")

✅ Streaming DataFrame created. Schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)

   isStreaming = True


In [0]:
# ── Transform: Enrich streaming orders ───────────────────────
# We add derived columns and clean the data before writing to Delta.
#
# Transformations applied:
#   1. Derive date_key (YYYYMMDD integer) from order_date string
#   2. Derive location_key by joining with dim_customer
#      (customer → country_code = location_key in dim_location)
#   3. Derive total_amount = quantity * unit_price (from dim_product)
#   4. Normalize status to lowercase
#
# Note: Streaming DataFrames support joins with static DataFrames.
# We read the static dimension Delta Tables as regular batch DataFrames
# and broadcast-join them onto the stream.

# Read dimension tables as static (batch) DataFrames for joining
static_customer = spark.read.format("delta").load(DELTA_CUSTOMER)
static_product  = spark.read.format("delta").load(DELTA_PRODUCT)

# Transform the stream
stream_enriched = (
    stream_raw
    # Join with dim_customer to get country_code (= location_key)
    .join(F.broadcast(static_customer.select("customer_id", "country_code")),
          on="customer_id", how="left")
    # Join with dim_product to get unit_price
    .join(F.broadcast(static_product.select("product_id", "unit_price")),
          on="product_id", how="left")
    # Derive date_key from order_date string
    .withColumn("date_key",
        F.date_format(F.to_date("order_date", "yyyy-MM-dd"), "yyyyMMdd").cast(IntegerType())
    )
    # Derive total_amount
    .withColumn("total_amount",
        F.round(F.col("quantity") * F.col("unit_price"), 2)
    )
    # Rename country_code to location_key (matches dim_location PK)
    .withColumnRenamed("country_code", "location_key")
    # Normalize status
    .withColumn("status", F.lower(F.trim(F.col("status"))))
    # Select final fact table columns in order
    .select(
        "order_id", "date_key", "customer_id", "product_id",
        "location_key", "quantity", "unit_price", "total_amount", "status"
    )
)

print("✅ Stream transformation defined. Enriched schema:")
stream_enriched.printSchema()

✅ Stream transformation defined. Enriched schema:
root
 |-- order_id: integer (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- location_key: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- status: string (nullable = true)



---
## 💾 STEP 5 — Load: Write Streaming Fact Table to Delta Lake

### `writeStream` vs `write`

A streaming DataFrame **cannot** use `.write` — it must use `.writeStream` which:
- Continuously monitors the source for new data
- Processes each micro-batch and appends to the Delta sink
- Maintains a **checkpoint** directory to track which data has been processed
  (so if the cluster restarts, it picks up exactly where it left off — no duplicates)

### `trigger(availableNow=True)`

This trigger mode tells Spark:
> *"Process all data that is currently available, then stop."*

This is ideal for our scenario — we want to process the uploaded CSV immediately rather than run a continuously-alive stream (which would consume cluster resources indefinitely). When a new CSV file is uploaded later, we simply re-run this cell.

### `outputMode("append")`

New orders are only ever added — we never update or delete from the fact table. `append` mode is correct here. For aggregations that update (e.g. running totals), you would use `update` or `complete` mode.


In [0]:
# Clear checkpoint to allow fresh streaming run
dbutils.fs.rm(BRONZE_CHECKPOINT, recurse=True)
dbutils.fs.rm(CHECKPOINT_PATH, recurse=True)
print("✅ Checkpoints cleared")

✅ Checkpoints cleared


In [0]:
# ── Register all Delta Tables as Spark SQL views ──────────────
# This lets us query them with spark.sql() using plain SQL syntax.

spark.read.format("delta").load(DELTA_ORDERS).createOrReplaceTempView("fact_orders")
spark.read.format("delta").load(DELTA_CUSTOMER).createOrReplaceTempView("dim_customer")
spark.read.format("delta").load(DELTA_PRODUCT).createOrReplaceTempView("dim_product")
spark.read.format("delta").load(DELTA_DATE).createOrReplaceTempView("dim_date")
spark.read.format("delta").load(DELTA_LOCATION).createOrReplaceTempView("dim_location")

print("✅ All Delta Tables registered as Spark SQL temp views:")
for t in ["fact_orders", "dim_customer", "dim_product", "dim_date", "dim_location"]:
    count = spark.sql(f"SELECT COUNT(*) AS n FROM {t}").collect()[0]["n"]
    print(f"   {t:<20} → {count} rows")

✅ All Delta Tables registered as Spark SQL temp views:
   fact_orders          → 30 rows
   dim_customer         → 5 rows
   dim_product          → 5 rows
   dim_date             → 366 rows
   dim_location         → 5 rows


---
## 📊 STEP 6 — Analytical SQL Queries Over the Data Mart

With all five tables registered as Spark SQL views, we can run standard SQL star-schema queries.
Each query joins `fact_orders` to one or more dimension tables to answer a specific business question.

The queries below demonstrate the full star schema — every dimension table is used at least once.


In [0]:
# ── Query 1: Revenue by Product Category ─────────────────────
# Business question: Which product categories generate the most revenue from completed orders?
# Joins: fact_orders → dim_product

print("━" * 60)
print("Query 1: Revenue by Product Category (completed orders only)")
print("━" * 60)

q1 = spark.sql("""
    SELECT
        p.category,
        COUNT(f.order_id)           AS total_orders,
        SUM(f.quantity)             AS units_sold,
        ROUND(SUM(f.total_amount), 2) AS total_revenue
    FROM fact_orders f
    JOIN dim_product p ON f.product_id = p.product_id
    WHERE f.status = 'completed'
    GROUP BY p.category
    ORDER BY total_revenue DESC
""")
q1.show(truncate=False)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Query 1: Revenue by Product Category (completed orders only)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
+-----------+------------+----------+-------------+
|category   |total_orders|units_sold|total_revenue|
+-----------+------------+----------+-------------+
|Books      |6           |22        |879.78       |
|Electronics|5           |12        |695.88       |
|Kitchen    |2           |8         |399.92       |
|Sports     |1           |5         |129.95       |
+-----------+------------+----------+-------------+



In [0]:
# ── Query 2: Monthly Revenue Trend ───────────────────────────
# Business question: How did revenue trend month-over-month in 2024?
# Joins: fact_orders → dim_date

print("━" * 60)
print("Query 2: Monthly Revenue Trend (completed orders)")
print("━" * 60)

q2 = spark.sql("""
    SELECT
        d.year,
        d.month,
        d.month_name,
        COUNT(f.order_id)             AS order_count,
        ROUND(SUM(f.total_amount), 2) AS monthly_revenue
    FROM fact_orders f
    JOIN dim_date d ON f.date_key = d.date_key
    WHERE f.status = 'completed'
    GROUP BY d.year, d.month, d.month_name
    ORDER BY d.year, d.month
""")
q2.show(truncate=False)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Query 2: Monthly Revenue Trend (completed orders)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
+----+-----+----------+-----------+---------------+
|year|month|month_name|order_count|monthly_revenue|
+----+-----+----------+-----------+---------------+
|2024|1    |January   |5          |549.85         |
|2024|2    |February  |3          |689.87         |
|2024|3    |March     |6          |865.81         |
+----+-----+----------+-----------+---------------+



In [0]:
# ── Query 3: Revenue by World Region ────────────────────────
# Business question: Which global regions drive the most non-refunded revenue?
# Joins: fact_orders → dim_location

print("━" * 60)
print("Query 3: Revenue by World Region (excluding refunded)")
print("━" * 60)

q3 = spark.sql("""
    SELECT
        l.region,
        l.country_name,
        COUNT(f.order_id)             AS orders,
        ROUND(SUM(f.total_amount), 2) AS revenue
    FROM fact_orders f
    JOIN dim_location l ON f.location_key = l.country_code
    WHERE f.status != 'refunded'
    GROUP BY l.region, l.country_name
    ORDER BY revenue DESC
""")
q3.show(truncate=False)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Query 3: Revenue by World Region (excluding refunded)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
+--------+--------------+------+-------+
|region  |country_name  |orders|revenue|
+--------+--------------+------+-------+
|Americas|United States |7     |985.79 |
|Asia    |Japan         |6     |875.81 |
|Asia    |China         |5     |809.82 |
|Americas|Mexico        |3     |415.93 |
|Europe  |United Kingdom|3     |371.9  |
+--------+--------------+------+-------+



In [0]:
# ── Query 4: Top Customers by Lifetime Value ─────────────────
# Business question: Who are our highest-value customers (completed orders)?
# Joins: fact_orders → dim_customer

print("━" * 60)
print("Query 4: Top 5 Customers by Lifetime Value")
print("━" * 60)

q4 = spark.sql("""
    SELECT
        c.name,
        c.country_code,
        COUNT(f.order_id)             AS completed_orders,
        ROUND(SUM(f.total_amount), 2) AS lifetime_value
    FROM fact_orders f
    JOIN dim_customer c ON f.customer_id = c.customer_id
    WHERE f.status = 'completed'
    GROUP BY c.customer_id, c.name, c.country_code
    ORDER BY lifetime_value DESC
    LIMIT 5
""")
q4.show(truncate=False)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Query 4: Top 5 Customers by Lifetime Value
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
+-------------+------------+----------------+--------------+
|name         |country_code|completed_orders|lifetime_value|
+-------------+------------+----------------+--------------+
|Alice Johnson|US          |4               |609.89        |
|Yuki Tanaka  |JP          |4               |525.88        |
|Chen Wei     |CN          |3               |499.87        |
|Bob Smith    |GB          |2               |319.92        |
|Maria Garcia |MX          |1               |149.97        |
+-------------+------------+----------------+--------------+



In [0]:
# ── Query 5: Weekend vs Weekday Performance ───────────────────
# Business question: Do customers order more (or spend more) on weekends vs weekdays?
# Joins: fact_orders → dim_date

print("━" * 60)
print("Query 5: Weekend vs Weekday Order Performance")
print("━" * 60)

q5 = spark.sql("""
    SELECT
        CASE d.is_weekend WHEN 1 THEN 'Weekend' ELSE 'Weekday' END AS day_type,
        COUNT(f.order_id)             AS total_orders,
        ROUND(AVG(f.total_amount), 2) AS avg_order_value,
        ROUND(SUM(f.total_amount), 2) AS total_revenue
    FROM fact_orders f
    JOIN dim_date d ON f.date_key = d.date_key
    GROUP BY d.is_weekend
    ORDER BY d.is_weekend
""")
q5.show(truncate=False)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Query 5: Weekend vs Weekday Order Performance
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
+--------+------------+---------------+-------------+
|day_type|total_orders|avg_order_value|total_revenue|
+--------+------------+---------------+-------------+
|Weekday |20          |175.06         |3501.29      |
|Weekend |10          |110.78         |1107.75      |
+--------+------------+---------------+-------------+



In [0]:
# ── Query 6: Full Star Schema Join — Order Detail Report ──────
# Business question: Show a denormalized order detail report using ALL 5 tables.
# This query demonstrates the complete star schema working end-to-end.

print("━" * 60)
print("Query 6: Full Star Schema — Denormalized Order Detail")
print("━" * 60)

q6 = spark.sql("""
    SELECT
        f.order_id,
        d.full_date       AS order_date,
        d.day_of_week,
        c.name            AS customer_name,
        l.country_name,
        l.region,
        p.product_name,
        p.category,
        f.quantity,
        f.unit_price,
        f.total_amount,
        f.status
    FROM fact_orders f
    JOIN dim_date     d ON f.date_key     = d.date_key
    JOIN dim_customer c ON f.customer_id  = c.customer_id
    JOIN dim_location l ON f.location_key = l.country_code
    JOIN dim_product  p ON f.product_id   = p.product_id
    ORDER BY f.order_id
""")
print(f"Total rows in full join: {q6.count()}")
q6.show(30, truncate=False)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Query 6: Full Star Schema — Denormalized Order Detail
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Total rows in full join: 30
+--------+----------+-----------+-------------+--------------+--------+-------------------+-----------+--------+----------+------------+---------+
|order_id|order_date|day_of_week|customer_name|country_name  |region  |product_name       |category   |quantity|unit_price|total_amount|status   |
+--------+----------+-----------+-------------+--------------+--------+-------------------+-----------+--------+----------+------------+---------+
|1       |2024-02-01|Thursday   |Alice Johnson|United States |Americas|Wireless Headphones|Electronics|3       |79.99     |239.97      |completed|
|2       |2024-01-12|Friday     |Bob Smith    |United Kingdom|Europe  |Wireless Headphones|Electronics|5       |79.99     |399.95      |refunded |
|3       |2024-01-28|Sunday     |Alice Johnson|United States 

---
## ✅ STEP 7 — Delta Lake Features: Time Travel & Table History

Delta Lake records every write operation in its transaction log (`_delta_log/`).  
This enables **Time Travel** — querying the table as it existed at a previous version or timestamp.

This is a key advantage of Delta Tables over plain Parquet files and is worth demonstrating to show mastery of the technology.


In [0]:
from delta.tables import DeltaTable

# ── Show transaction history of fact_orders ───────────────────
delta_fact = DeltaTable.forPath(spark, DELTA_ORDERS)

print("━" * 60)
print("Delta Table History: fact_orders")
print("━" * 60)
delta_fact.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

# ── Time Travel: Query version 0 (initial load) ───────────────
print("━" * 60)
print("Time Travel: fact_orders at version 0")
print("━" * 60)
(
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .load(DELTA_ORDERS)
    .agg({"order_id": "count"})
    .withColumnRenamed("count(order_id)", "row_count_at_v0")
    .show()
)

print("\n✅ Delta Time Travel verified — transaction log is intact.")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Delta Table History: fact_orders
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
+-------+-------------------+----------------+------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation       |operationParameters                                                                                         |
+-------+-------------------+----------------+------------------------------------------------------------------------------------------------------------+
|1      |2026-05-08 22:43:35|STREAMING UPDATE|{outputMode -> Append, queryId -> 80011c8f-a2d0-4fa7-83d4-63b9e88d3476, epochId -> 0, statsOnLoad -> false} |
|0      |2026-05-08 22:43:33|STREAMING UPDATE|{outputMode -> Append, queryId -> 80011c8f-a2d0-4fa7-83d4-63b9e88d3476, epochId -> -1, statsOnLoad -> false}|
+-------+-------------------+----------------+-------------------

## Pipeline Summary

| Stage | Technology | Output |
|---|---|---|
| Relational DB source | SQLite simulating Azure SQL OLTP | dim_customer Delta Table |
| NoSQL source | MongoDB Atlas | dim_product Delta Table |
| File system source | CSV from Databricks Volume | dim_location Delta Table |
| Derived dimension | PySpark calendar generation | dim_date Delta Table |
| Streaming source | 3 JSON files via Spark AutoLoader | Bronze fact_orders Delta Table |
| Silver transform | Enriched fact data joined with SQL customer, MongoDB product, CSV location, derived date | 30 rows |
| Gold aggregation | Spark SQL business queries | 3 analytical Delta Tables |
| Auditability | Delta Time Travel | Full version history |

**Total tables in data mart:** 5 (1 fact + 4 dimensions)
**Data sources:** Relational DB (SQLite) + NoSQL (MongoDB) + File System (CSV) + Derived
**Streaming guarantees:** exactly-once delivery via Delta Lake checkpointing

In [0]:
import json, random
from datetime import datetime, timedelta

random.seed(42)
base_date = datetime(2024, 1, 1)

orders = []
for i in range(1, 31):
    orders.append({
        "order_id":    i,
        "customer_id": random.randint(1,5),
        "product_id":  random.randint(1,5),
        "quantity":    random.randint(1,5),
        "order_date":  (base_date + timedelta(days=random.randint(0,90))).strftime('%Y-%m-%d'),
        "status":      random.choice(['completed','completed','pending','refunded'])
    })

# Split into 3 batches simulating 3 streaming intervals
batch1 = orders[0:10]   # orders 1-10
batch2 = orders[10:20]  # orders 11-20
batch3 = orders[20:30]  # orders 21-30

volume_path = "/Volumes/workspace/default/orders_data"

for i, batch in enumerate([batch1, batch2, batch3], 1):
    path = f"{volume_path}/stream_batch_{i}.json"
    with open(path, 'w') as f:
        for record in batch:
            f.write(json.dumps(record) + '\n')  # newline-delimited JSON
    print(f"✅ stream_batch_{i}.json written — {len(batch)} records")

# Verify
files = [f for f in dbutils.fs.ls(f"dbfs:/Volumes/workspace/default/orders_data/") 
         if 'batch' in f.name]
print(f"\n{len(files)} JSON batch files ready for AutoLoader")

✅ stream_batch_1.json written — 10 records
✅ stream_batch_2.json written — 10 records
✅ stream_batch_3.json written — 10 records

3 JSON batch files ready for AutoLoader


In [0]:
# ============================================================
# BRONZE / SILVER / GOLD ARCHITECTURE
# Bronze: Raw streaming data (AutoLoader)
# Silver: Cleaned + joined with dimensions
# Gold:   Aggregated business-ready tables
# ============================================================

STREAM_SOURCE  = "/Volumes/workspace/default/orders_data/"
BRONZE_PATH    = "/Volumes/workspace/default/orders_data/delta/bronze/fact_orders"
SILVER_PATH    = "/Volumes/workspace/default/orders_data/delta/silver/fact_orders"
GOLD_PATH      = "/Volumes/workspace/default/orders_data/delta/gold/"
BRONZE_CHECKPOINT = "/Volumes/workspace/default/orders_data/checkpoints/bronze"

print("✅ Bronze/Silver/Gold paths configured.")
print(f"   Source  : {STREAM_SOURCE}")
print(f"   Bronze  : {BRONZE_PATH}")
print(f"   Silver  : {SILVER_PATH}")
print(f"   Gold    : {GOLD_PATH}")

✅ Bronze/Silver/Gold paths configured.
   Source  : /Volumes/workspace/default/orders_data/
   Bronze  : /Volumes/workspace/default/orders_data/delta/bronze/fact_orders
   Silver  : /Volumes/workspace/default/orders_data/delta/silver/fact_orders
   Gold    : /Volumes/workspace/default/orders_data/delta/gold/


In [0]:
import sqlite3, pandas as pd, csv

# --- SQLite → dim_customer ---
conn = sqlite3.connect('/tmp/oltp_source.db')
cur = conn.cursor()
cur.executescript("""
    DROP TABLE IF EXISTS customers;
    CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, email TEXT, country_code TEXT);
""")
cur.executemany("INSERT INTO customers VALUES (?,?,?,?)", [
    (1,'Alice Johnson','alice@example.com','US'),
    (2,'Bob Smith','bob@example.com','GB'),
    (3,'Chen Wei','chen@example.com','CN'),
    (4,'Maria Garcia','maria@example.com','MX'),
    (5,'Yuki Tanaka','yuki@example.com','JP'),
])
conn.commit()
df_customers_sql = spark.createDataFrame(pd.read_sql("SELECT * FROM customers", conn))
conn.close()
df_customers_sql.write.format("delta").mode("overwrite").option("overwriteSchema","true").save(DELTA_CUSTOMER)
print(f"✅ dim_customer from SQLite: {df_customers_sql.count()} rows")

# --- CSV → dim_location ---
with open("/Volumes/workspace/default/orders_data/dim_location.csv","w",newline="") as f:
    csv.writer(f).writerows([
        ["country_code","country_name","region","capital"],
        ["US","United States","Americas","Washington D.C."],
        ["GB","United Kingdom","Europe","London"],
        ["CN","China","Asia","Beijing"],
        ["MX","Mexico","Americas","Mexico City"],
        ["JP","Japan","Asia","Tokyo"],
    ])
df_location_csv = spark.read.option("header","true").csv("/Volumes/workspace/default/orders_data/dim_location.csv")
df_location_csv.write.format("delta").mode("overwrite").option("overwriteSchema","true").save(DELTA_LOCATION)
print(f"✅ dim_location from CSV: {df_location_csv.count()} rows")

# Clear checkpoints
dbutils.fs.rm(BRONZE_CHECKPOINT, recurse=True)
dbutils.fs.rm(CHECKPOINT_PATH, recurse=True)
print("✅ Checkpoints cleared — ready for Bronze AutoLoader")

✅ dim_customer from SQLite: 5 rows
✅ dim_location from CSV: 5 rows
✅ Checkpoints cleared — ready for Bronze AutoLoader


In [0]:
# Clear Bronze Delta to remove duplicate records from multiple runs
dbutils.fs.rm(BRONZE_PATH, recurse=True)
dbutils.fs.rm(BRONZE_CHECKPOINT, recurse=True)
print("✅ Bronze cleared")

✅ Bronze cleared


In [0]:
# ============================================================
# BRONZE LAYER — AutoLoader (Spark AutoLoader / cloudFiles)
# Reads raw JSON files as they arrive — simulates real-time streaming
# AutoLoader automatically detects new files using cloudFiles format
# ============================================================
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

orders_schema = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("product_id",  IntegerType(), False),
    StructField("quantity",    IntegerType(), False),
    StructField("order_date",  StringType(),  False),
    StructField("status",      StringType(),  False),
])

bronze_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.inferColumnTypes", "false")
    .option("cloudFiles.schemaLocation", BRONZE_CHECKPOINT + "/schema")
    .schema(orders_schema)
    .load(STREAM_SOURCE + "*.json")
)

bronze_query = (
    bronze_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", BRONZE_CHECKPOINT)
    .trigger(availableNow=True)
    .start(BRONZE_PATH)
)

bronze_query.awaitTermination()
bronze_df = spark.read.format("delta").load(BRONZE_PATH)
print(f"✅ BRONZE layer loaded — {bronze_df.count()} raw records")
bronze_df.show(5, truncate=False)

✅ BRONZE layer loaded — 30 raw records
+--------+-----------+----------+--------+----------+---------+
|order_id|customer_id|product_id|quantity|order_date|status   |
+--------+-----------+----------+--------+----------+---------+
|21      |5          |4         |5       |2024-02-21|pending  |
|22      |2          |2         |5       |2024-03-04|completed|
|23      |1          |1         |2       |2024-03-21|completed|
|24      |4          |5         |1       |2024-02-19|refunded |
|25      |5          |4         |5       |2024-02-02|completed|
+--------+-----------+----------+--------+----------+---------+
only showing top 5 rows


## Dimension Tables — Multi-Source Integration
Before Silver transformation, all dimension Delta Tables are populated from their respective sources:
- dim_customer: SQLite relational DB → Delta
- dim_location: CSV file → Delta
- dim_product: MongoDB Atlas → Delta (populated in Step 2 above)

In [0]:
# Re-confirm all dimension Delta Tables are populated from correct sources
# dim_customer: from SQLite relational source
# dim_product: from MongoDB Atlas NoSQL source  
# dim_location: from CSV file source

print("Dimension table row counts before Silver join:")
print(f"  dim_customer (SQL source)     : {spark.read.format('delta').load(DELTA_CUSTOMER).count()} rows")
print(f"  dim_product  (MongoDB source) : {spark.read.format('delta').load(DELTA_PRODUCT).count()} rows")
print(f"  dim_location (CSV source)     : {spark.read.format('delta').load(DELTA_LOCATION).count()} rows")
print(f"  dim_date     (derived)        : {spark.read.format('delta').load(DELTA_DATE).count()} rows")
print(f"  Bronze fact  (JSON streaming) : {spark.read.format('delta').load(BRONZE_PATH).count()} rows")
print("\n✅ All sources confirmed — proceeding to Silver transformation")

Dimension table row counts before Silver join:
  dim_customer (SQL source)     : 5 rows
  dim_product  (MongoDB source) : 5 rows
  dim_location (CSV source)     : 5 rows
  dim_date     (derived)        : 366 rows
  Bronze fact  (JSON streaming) : 30 rows

✅ All sources confirmed — proceeding to Silver transformation


In [0]:
# ============================================================
# SILVER LAYER — Clean + Join with Dimension Tables
# Enriches raw Bronze data with reference data from dimensions
# This is where fact and dimension tables are joined
# ============================================================

import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType

# Read Bronze Delta Table (batch read for Silver transformation)
bronze_df = spark.read.format("delta").load(BRONZE_PATH)

# # Read dimension tables from Delta (populated from multiple sources: SQL, MongoDB, CSV)
dim_customer = spark.read.format("delta").load(DELTA_CUSTOMER)
dim_product  = spark.read.format("delta").load(DELTA_PRODUCT)
dim_location = spark.read.format("delta").load(DELTA_LOCATION)
dim_date     = spark.read.format("delta").load(DELTA_DATE)

# Transform: derive date_key and total_amount, join all dimensions
silver_df = (
    bronze_df
    # Join with dim_customer to get country_code (location_key)
    .join(F.broadcast(dim_customer.select("customer_id", "country_code")),
          on="customer_id", how="left")
    # Join with dim_product to get unit_price
    .join(F.broadcast(dim_product.select("product_id", "unit_price")),
          on="product_id", how="left")
    # Derive date_key (YYYYMMDD integer)
    .withColumn("date_key",
        F.date_format(F.to_date("order_date", "yyyy-MM-dd"), "yyyyMMdd").cast(IntegerType())
    )
    # Derive total_amount
    .withColumn("total_amount",
        F.round(F.col("quantity") * F.col("unit_price"), 2)
    )
    # Rename and normalize
    .withColumnRenamed("country_code", "location_key")
    .withColumn("status", F.lower(F.trim(F.col("status"))))
    .select("order_id", "date_key", "customer_id", "product_id",
            "location_key", "quantity", "unit_price", "total_amount", "status")
)

# Write to Silver Delta Table
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)

silver_count = spark.read.format("delta").load(SILVER_PATH).count()
print(f"✅ SILVER layer loaded — {silver_count} enriched records")
spark.read.format("delta").load(SILVER_PATH).show(5, truncate=False)

✅ SILVER layer loaded — 30 enriched records
+--------+--------+-----------+----------+------------+--------+----------+------------+---------+
|order_id|date_key|customer_id|product_id|location_key|quantity|unit_price|total_amount|status   |
+--------+--------+-----------+----------+------------+--------+----------+------------+---------+
|21      |20240221|5          |4         |JP          |5       |49.99     |249.95      |pending  |
|22      |20240304|2          |2         |GB          |5       |39.99     |199.95      |completed|
|23      |20240321|1          |1         |US          |2       |79.99     |159.98      |completed|
|24      |20240219|4          |5         |MX          |1       |35.99     |35.99       |refunded |
|25      |20240202|5          |4         |JP          |5       |49.99     |249.95      |completed|
+--------+--------+-----------+----------+------------+--------+----------+------------+---------+
only showing top 5 rows


In [0]:
# Register all dimension tables as Spark SQL temp views for Gold queries
spark.read.format("delta").load(DELTA_CUSTOMER).createOrReplaceTempView("dim_customer")
spark.read.format("delta").load(DELTA_PRODUCT).createOrReplaceTempView("dim_product")
spark.read.format("delta").load(DELTA_LOCATION).createOrReplaceTempView("dim_location")
spark.read.format("delta").load(DELTA_DATE).createOrReplaceTempView("dim_date")
spark.read.format("delta").load(SILVER_PATH).createOrReplaceTempView("silver_orders")

print("✅ All tables registered as Spark SQL temp views")

✅ All tables registered as Spark SQL temp views


In [0]:
# ============================================================
# GOLD LAYER — Aggregated, business-ready analytical tables
# Gold tables are optimized for reporting and BI queries
# Each Gold table answers a specific business question
# ============================================================

silver = spark.read.format("delta").load(SILVER_PATH)
silver.createOrReplaceTempView("silver_orders")

# --- Gold Table 1: Revenue by Product Category ---
gold_category = spark.sql("""
    SELECT
        p.category,
        COUNT(s.order_id)             AS total_orders,
        SUM(s.quantity)               AS units_sold,
        ROUND(SUM(s.total_amount), 2) AS total_revenue
    FROM silver_orders s
    JOIN dim_product p ON s.product_id = p.product_id
    WHERE s.status = 'completed'
    GROUP BY p.category
    ORDER BY total_revenue DESC
""")
gold_category.write.format("delta").mode("overwrite").save(GOLD_PATH + "revenue_by_category")
print("✅ Gold Table 1: Revenue by Category")
gold_category.show(truncate=False)

# --- Gold Table 2: Monthly Revenue Trend ---
gold_monthly = spark.sql("""
    SELECT
        d.year, d.month, d.month_name,
        COUNT(s.order_id)             AS order_count,
        ROUND(SUM(s.total_amount), 2) AS monthly_revenue
    FROM silver_orders s
    JOIN dim_date d ON s.date_key = d.date_key
    WHERE s.status = 'completed'
    GROUP BY d.year, d.month, d.month_name
    ORDER BY d.year, d.month
""")
gold_monthly.write.format("delta").mode("overwrite").save(GOLD_PATH + "revenue_by_month")
print("✅ Gold Table 2: Monthly Revenue Trend")
gold_monthly.show(truncate=False)

# --- Gold Table 3: Revenue by Region ---
gold_region = spark.sql("""
    SELECT
        l.region, l.country_name,
        COUNT(s.order_id)             AS orders,
        ROUND(SUM(s.total_amount), 2) AS revenue
    FROM silver_orders s
    JOIN dim_location l ON s.location_key = l.country_code
    WHERE s.status != 'refunded'
    GROUP BY l.region, l.country_name
    ORDER BY revenue DESC
""")
gold_region.write.format("delta").mode("overwrite").save(GOLD_PATH + "revenue_by_region")
print("✅ Gold Table 3: Revenue by Region")
gold_region.show(truncate=False)

print("\n✅ GOLD layer complete — 3 analytical tables written to Delta Lake")

✅ Gold Table 1: Revenue by Category
+-----------+------------+----------+-------------+
|category   |total_orders|units_sold|total_revenue|
+-----------+------------+----------+-------------+
|Books      |6           |22        |879.78       |
|Electronics|5           |12        |695.88       |
|Kitchen    |2           |8         |399.92       |
|Sports     |1           |5         |129.95       |
+-----------+------------+----------+-------------+

✅ Gold Table 2: Monthly Revenue Trend
+----+-----+----------+-----------+---------------+
|year|month|month_name|order_count|monthly_revenue|
+----+-----+----------+-----------+---------------+
|2024|1    |January   |5          |549.85         |
|2024|2    |February  |3          |689.87         |
|2024|3    |March     |6          |865.81         |
+----+-----+----------+-----------+---------------+

✅ Gold Table 3: Revenue by Region
+--------+--------------+------+-------+
|region  |country_name  |orders|revenue|
+--------+--------------+---

## Data Source Integration Summary

This pipeline demonstrates extraction from multiple source system types and integrates them into a dimensional Delta Lakehouse.

| Source | Technology | Final Delta Table |
|---|---|---|
| Relational DB / OLTP source | SQLite simulating Azure SQL | dim_customer |
| NoSQL document store | MongoDB Atlas | dim_product |
| Cloud file system | CSV file stored in Databricks Volume | dim_location |
| Derived pipeline dimension | PySpark calendar generation | dim_date |
| Streaming file source | 3 JSON files processed by Spark AutoLoader | Bronze fact_orders |

## Bronze / Silver / Gold Architecture

| Layer | Description | Record Count |
|---|---|---|
| Bronze | Raw JSON streaming fact data ingested with Spark AutoLoader | 30 rows |
| Silver | Cleaned and enriched fact data joined with SQL customer, MongoDB product, CSV location, and derived date dimensions | 30 rows |
| Gold | Aggregated analytical tables for business reporting | 3 tables |

### File-Based Dimension Source: dim_location

To satisfy the file-system source requirement, I created a static location dimension from a CSV file stored in a Databricks Volume. This CSV file was then read into Spark and written as a Delta table named `dim_location`.

This dimension is later joined with streaming fact order data during the Silver table phase using `country_code` as the business key.